# 1. Setup OpenMeteo Data Source

In [ ]:
%pip install openmeteo-requests

In [ ]:
%pip install requests-cache retry-requests numpy pandas

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"hourly": "temperature_2m",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


In [ ]:
# Đọc danh sách đơn vị hành chính và tọa độ từ Parquet.
from pathlib import Path

# Hỗ trợ chạy notebook từ thư mục gốc hoặc trực tiếp từ thư mục data/.
parquet_candidates = [Path("data/dien_bien_locations.parquet"), Path("dien_bien_locations.parquet")]
locations_path = next((path for path in parquet_candidates if path.exists()), None)
if locations_path is None:
    raise FileNotFoundError("Không tìm thấy data/dien_bien_locations.parquet")

locations_df = pd.read_parquet(locations_path).rename(
    columns={"new_admin_unit": "admin_unit", "old_admin_unit": "admin_unit_old"}
)

locations = locations_df.to_dict(orient="records")
print(f"Đã đọc {len(locations)} địa điểm từ {locations_path}")

weather_params = {
    "latitude": [place["latitude"] for place in locations],
    "longitude": [place["longitude"] for place in locations],
    "hourly": [
        "temperature_2m", "relative_humidity_2m", "precipitation",
        "weather_code", "wind_speed_10m", "visibility",
    ],
    "timezone": "Asia/Ho_Chi_Minh",
    "forecast_days": 7,
}

responses = openmeteo.weather_api(url, params=weather_params)
frames = []
variables = weather_params["hourly"]

for place, response in zip(locations, responses):
    hourly = response.Hourly()
    frame = pd.DataFrame({
        "time": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        ).tz_convert("Asia/Ho_Chi_Minh"),
        **{name: hourly.Variables(index).ValuesAsNumpy() for index, name in enumerate(variables)},
    })
    frame.insert(0, "admin_unit", place["admin_unit"])
    frame.insert(1, "admin_unit_old", place["admin_unit_old"])
    frame.insert(2, "province", "Điện Biên")
    frame["latitude"] = response.Latitude()
    frame["longitude"] = response.Longitude()
    frames.append(frame)

dien_bien_weather_df = (
    pd.concat(frames, ignore_index=True)
    .sort_values(["time", "admin_unit"], ignore_index=True)
)

# Xem cùng một thời điểm ở tất cả điểm để xác nhận dữ liệu không bị lặp một địa danh.\n
first_time = dien_bien_weather_df["time"].min()
display(dien_bien_weather_df.loc[dien_bien_weather_df["time"] == first_time])
print(f"{len(dien_bien_weather_df):,} dòng | {dien_bien_weather_df['admin_unit'].nunique()} xã/phường")


In [ ]:
dien_bien_weather_df.dtypes

In [ ]:
display(dien_bien_weather_df.head(200))

In [ ]:
print("Bắt đầu:", dien_bien_weather_df["time"].min())
print("Kết thúc:", dien_bien_weather_df["time"].max())
print("Số mốc thời gian:", dien_bien_weather_df["time"].nunique())
print("Số địa điểm:", dien_bien_weather_df["admin_unit_old"].nunique())
print("Tổng số dòng:", len(dien_bien_weather_df))

In [ ]:
# Tải 5 năm dữ liệu lịch sử ERA5 theo giờ và lưu Parquet theo năm/quý.
%pip install pyarrow requests

import subprocess
import sys
from pathlib import Path
import pyarrow.dataset as ds

location_candidates = [
    Path("data/dien_bien_locations.parquet"),
    Path("dien_bien_locations.parquet"),
]
locations_path = next((path for path in location_candidates if path.exists()), None)
if locations_path is None:
    raise FileNotFoundError("Không tìm thấy dien_bien_locations.parquet")

script_candidates = [
    Path("data/download_historical_weather.py"),
    Path("download_historical_weather.py"),
]
download_script = next((path for path in script_candidates if path.exists()), None)
if download_script is None:
    raise FileNotFoundError("Không tìm thấy download_historical_weather.py")

history_dir = locations_path.parent / "weather_history"
subprocess.run(
    [
        sys.executable,
        str(download_script),
        "--locations", str(locations_path),
        "--output", str(history_dir),
        "--start-year", "2021",
        "--end-year", "2025",
        "--batch-size", "10",
        "--request-delay", "60",
    ],
    check=True,
)

# Đọc thử 100 dòng mà không nạp toàn bộ 3,73 triệu dòng vào RAM.
history_dataset = ds.dataset(
    history_dir,
    format="parquet",
    partitioning="hive",
)
history_preview = history_dataset.head(100).to_pandas()
location_lookup = pd.read_parquet(locations_path)
history_preview = history_preview.merge(location_lookup, on="location_id", how="left")
display(history_preview)
